# Tests: `fasterai.distill.distillation_callback` (source `nbs/distill/distillation_callback.ipynb`)

In [ ]:
from fastcore.test import *
from collections import OrderedDict
import torch
import torch.nn as nn
from fastai.learner import Learner
from fasterai.distill.distillation_callback import *

In [ ]:
# match_feature_layers test
from torchvision.models import resnet18, resnet50
_student = resnet18(num_classes=10)
_teacher = resnet50(num_classes=10)
_x = torch.randn(1, 3, 64, 64)

_pairs = match_feature_layers(_student, _teacher, _x)
assert 'student' in _pairs and 'teacher' in _pairs
test_eq(len(_pairs['student']), len(_pairs['teacher']))
assert len(_pairs['student']) >= 3  # at least 3 resolution matches

# Verify spatial dims actually match
_s_hooks, _t_hooks, _s_shapes, _t_shapes = [], [], {}, {}
for name in _pairs['student']:
    m = dict(_student.named_modules())[name]
    _s_hooks.append(m.register_forward_hook(lambda mod, inp, out, n=name: _s_shapes.update({n: out.shape[2:]})))
for name in _pairs['teacher']:
    m = dict(_teacher.named_modules())[name]
    _t_hooks.append(m.register_forward_hook(lambda mod, inp, out, n=name: _t_shapes.update({n: out.shape[2:]})))
with torch.no_grad(): _student(_x); _teacher(_x)
for h in _s_hooks + _t_hooks: h.remove()

for s, t in zip(_pairs['student'], _pairs['teacher']):
    test_eq(_s_shapes[s], _t_shapes[t])  # spatial dims match

In [ ]:
from fastcore.test import *

def _test_model():
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(32, 10)
    )

model = _test_model()

# get_model_layers returns list of strings
layers = get_model_layers(model)
assert isinstance(layers, list)
assert all(isinstance(n, str) for n in layers)
assert len(layers) > 0

# get_model_layers with repr returns dict (OrderedDict)
layers_d = get_model_layers(model, get_layer_repr=True)
assert isinstance(layers_d, OrderedDict)
assert len(layers_d) > 0

# get_module_by_name returns correct module
m0 = get_module_by_name(model, '0')
test_is(m0, model[0])

# get_module_by_name returns None for nonexistent
test_eq(get_module_by_name(model, 'nonexistent'), None)

# KnowledgeDistillationCallback construction
teacher = _test_model()
from fasterai.distill.losses import SoftTarget
cb = KnowledgeDistillationCallback(
    teacher=teacher,
    loss=SoftTarget,
    weight=0.5
)
test_eq(cb.weight, 0.5)
test_eq(cb.current_weight, 0.5)
assert cb.activations_student is None

In [ ]:
#| slow
# Teacher-student training with KnowledgeDistillationCallback
from torch.utils.data import TensorDataset
from fastai.data.core import DataLoaders
from fasterai.distill.losses import SoftTarget

_teacher = nn.Sequential(
    nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10)
)
_student = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(8, 10)
)

_X = torch.randn(64, 3, 8, 8)
_y = torch.randint(0, 10, (64,))
_dls = DataLoaders.from_dsets(
    TensorDataset(_X[:48], _y[:48]),
    TensorDataset(_X[48:], _y[48:]),
    bs=16, device='cpu'
)

_cb = KnowledgeDistillationCallback(teacher=_teacher, loss=SoftTarget)
_learn = Learner(_dls, _student, loss_func=nn.CrossEntropyLoss(), cbs=[_cb])
_learn.fit(2)  # verify it runs end-to-end without error